In [1]:
import numpy as np
import os
from collections import defaultdict
from sklearn.decomposition import PCA
import jsonlines

In [2]:
formats = {
    "Completion": "",
    "triplet": "_triplet",
    "ODQA": "_ODQA",
    "MC": "_MC",
    "TF": "_TF",
    "YN": "_YN",
    # "Merge_all": "_memit_merge_MC_ODQA_TF_YN_completion_triplet"
}

def load_vectors_for_pca(model_name, case_id=0, v_num_grad_steps=25):
    data_dir = f"../../data/cache/kvs/{model_name}"

    data = defaultdict(dict)

    for fmt_name, fmt_suffix in formats.items():
        file_path = os.path.join(
            data_dir,
            f"multiformat_counterfact_1000_layer_7_clamp_0.75_case_{case_id}{fmt_suffix}_trajectory.npz"
        )
        gradient_descent_trajectory = np.load(file_path)["v_star"]
        data[fmt_name]['traj'] = gradient_descent_trajectory

    # key = f"base_{fmt_name}"
    # fmt_names = list(formats.keys())
    # for i, fmt_name in enumerate(fmt_names):
    #     for j, fmt_name2 in enumerate(fmt_names):
    #         if i < j:
    #             key = f"{fmt_name}_{fmt_name2}"
    #             linear_interpolation_trajectory = np.linspace(data[fmt_name]['traj'][-1], data[fmt_name2]['traj'][-1], v_num_grad_steps)
    #             data[key]['traj'] = linear_interpolation_trajectory
    
    ################################################
    # Load efficacy scores
    with jsonlines.open(f"../results/evaluation/{model_name[:-6]}/multiformat_counterfact_1000_MEMIT_v_star_trajectory.jsonl") as fin:
        for line in fin.iter():
            if line["case_id"] != case_id:
                continue
            key = line["traj_key"]
            if key == "Merge_all":
                continue
            if "traj_efficacy_magnitude" not in data[key]:
                for score_name in ["efficacy_magnitude", "triplet_efficacy_magnitude", "ODQA_efficacy_magnitude", "MC_efficacy_magnitude", "TF_efficacy_magnitude", "YN_efficacy_magnitude"]:
                    data[key]["traj_"+score_name] = []

            assert len(data[key]["traj_efficacy_magnitude"]) == line["traj_step"]
            
            for score_name in ["efficacy_magnitude", "triplet_efficacy_magnitude", "ODQA_efficacy_magnitude", "MC_efficacy_magnitude", "TF_efficacy_magnitude", "YN_efficacy_magnitude"]:
                data[key]["traj_"+score_name].append(line[score_name])
                # if score_name == "efficacy_magnitude":
                #     print(line[score_name])
    ################################################

    pca_points = []
    
    for fmt_name in formats:
        if "Completion" in fmt_name: # avoid duplicate
            pca_points.append(data[fmt_name]['traj'][0])
        pca_points.append(data[fmt_name]['traj'][-1])

    # PCA
    pca_points = np.stack(pca_points, axis=0)
    pca = PCA(n_components=2)
    pca.fit(pca_points)

    all_points_2d = []

    for data_key, trajectory in data.items():
        if data_key == "interval_points":
            continue
        if 'traj' not in trajectory:
            continue
        traj = np.array(trajectory['traj'])
        proj_2d = pca.transform(traj)
        for pt in proj_2d:
            all_points_2d.append(pt)
    all_points_2d = np.array(all_points_2d)

    grid_x, grid_y = np.mgrid[
        all_points_2d[:, 0].min():all_points_2d[:, 0].max():50j,
        all_points_2d[:, 1].min():all_points_2d[:, 1].max():50j
    ]

    # interval_points_2d = []
    # for i in range(50):
    #     for j in range(50):
    #         interval_points_2d.append([grid_x[i,j], grid_y[i,j]])
    # interval_points = pca.inverse_transform(np.array(interval_points_2d))
    # data["interval_points"]["traj"] = interval_points

    return data, pca

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe
from scipy.interpolate import griddata
import matplotlib.ticker as ticker


plt.rcParams.update({
    "font.size": 16,          
    "axes.labelsize": 16,     
    "axes.titlesize": 17,     
    "xtick.labelsize": 15,    
    "ytick.labelsize": 15,    
    "legend.fontsize": 15,    
})


def plot_pca_trajectories_with_interpolated_efficacy(data, pca, case_id, fmt_prefix=""):
    fig = plt.figure(figsize=(10 if fmt_prefix=="TF_" else 9, 7))
    color_map = {
        "Completion": "green",
        "triplet": "red",
        "ODQA": "blue",
        "MC": "purple",
        "TF": "orange",
        "YN": "cyan",
        # "Merge_all": "black"
    }

    # 1. Collect all points and corresponding efficacy scores
    all_points_2d = []
    all_efficacies = []

    for data_key, fmt_data in data.items():
        # if "_" in data_key and data_key not in ["interval_points", "Merge_all"]:
        #     continue
        # if "_" in data_key and data_key not in ["Merge_all"]:
        #     continue
        if "_" in data_key:
            continue
        for traj_key, eff_key in [("traj", f"traj_{fmt_prefix}efficacy_magnitude")]:
            traj = np.array(fmt_data[traj_key])
            effs = fmt_data.get(eff_key, [])

            proj_2d = pca.transform(traj)
            for i, pt in enumerate(proj_2d):
                try:
                    eff = effs[i]
                    all_points_2d.append(pt)
                    all_efficacies.append(eff)
                except:
                    break

    all_points_2d = np.array(all_points_2d)
    all_efficacies = np.array(all_efficacies)
    # print(max(all_efficacies))
    # print(data.keys())
    # print(all_efficacies)
    # print()

    if len(all_points_2d) == 0:
        print("No valid efficacy data found.")
        return

    # 2. Interpolate over a grid
    grid_x, grid_y = np.mgrid[
        all_points_2d[:, 0].min()*1.05:all_points_2d[:, 0].max()*1.05:1000j,
        all_points_2d[:, 1].min()*1.05:all_points_2d[:, 1].max()*1.05:1000j
    ]
    # method='nearest' 'linear' 'cubic'
    grid_z = griddata(all_points_2d, all_efficacies, (grid_x, grid_y), method='nearest')

    # 3. Plot interpolated heatmap
    im = plt.imshow(
        grid_z.T,
        extent=(grid_x.min(), grid_x.max(), grid_y.min(), grid_y.max()),
        origin='lower',
        cmap='coolwarm',
        aspect='auto',
        alpha=0.85,
        vmin=-1, vmax=1
    )

    # Draw boundary
    cs = plt.contour(
        grid_x, grid_y, grid_z,
        levels=[0],  # boundary between regions
        colors="k", linewidths=8.0
    )

    sm = plt.cm.ScalarMappable(cmap='coolwarm')
    sm.set_array(all_efficacies)

    # 4. Overlay trajectory start/end points
    for fmt_name, fmt_data in data.items():
        if fmt_name not in formats:
            continue
        color = color_map.get(fmt_name, 'gray')

        traj = np.array(fmt_data["traj"])
        traj_2d = pca.transform(traj)
        plt.plot(traj_2d[:, 0], traj_2d[:, 1], color=color, linestyle='-', linewidth=5, alpha=0.8, label=f'{fmt_name.replace("Completion", "completion")}')
        plt.scatter(traj_2d[1:-1, 0], traj_2d[1:-1, 1], marker='o', color=color, label=None, s=50, alpha=0.6)
        plt.scatter(traj_2d[0, 0], traj_2d[0, 1], marker='o', color=color, label=None, s=60)
        plt.scatter(traj_2d[-1, 0], traj_2d[-1, 1], marker='*', color=color, label=None, s=720)

    plt.title(f"PCA of v* Trajectories with Efficacy Landscape ({fmt_prefix[:-1] if fmt_prefix != '' else 'completion'})")
    plt.xlabel("PCA 1")
    plt.ylabel("PCA 2")
    plt.grid(False)
    if fmt_prefix == "TF_":
        cbar = plt.colorbar(im, label="Efficacy Magnitude")
        cbar.locator = ticker.MultipleLocator(0.5)
        cbar.formatter = ticker.FormatStrFormatter("%.1f")
        cbar.update_ticks()
        plt.legend()
    plt.tight_layout()
    
    save_dir = f"v_star_dynamics/case_{case_id}"
    os.makedirs(save_dir, exist_ok=True)
    save_fname = fmt_prefix[:-1] if fmt_prefix != "" else "completion"
    plt.savefig(os.path.join(save_dir, f'{save_fname}_refined.png'), bbox_inches='tight', dpi=300)
    plt.close()

    # plt.show()

In [4]:
model_name = "meta-llama_Llama-3.2-3B-Instruct_MEMIT"

for case_id in range(110):
    try:
        data, pca = load_vectors_for_pca(model_name, case_id)

        for fmt_prefix in ["", "TF_", "MC_", "ODQA_", "YN_", "triplet_"]:
            plot_pca_trajectories_with_interpolated_efficacy(data, pca, case_id, fmt_prefix=fmt_prefix)
    except Exception as e:
        print(e)

KeyboardInterrupt: 

<Figure size 900x700 with 0 Axes>